In [ ]:
# Optional libraries. These may take a few minutes in Colab.
import sys, subprocess, importlib.util
from pathlib import Path
import os, json, textwrap, warnings, random, time

warnings.filterwarnings("ignore")
def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {package_name} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

for pkg, imp in [('shap', 'shap'), ('lime', 'lime'), ('joblib', 'joblib')]:
    try:
        ensure_package(pkg, imp)
    except Exception as e:
        print(f'Optional package {pkg} could not be installed: {e}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error, explained_variance_score, max_error
from sklearn.inspection import permutation_importance
from scipy import stats
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', None)
print('Libraries loaded successfully.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, json, textwrap, warnings, random, time

warnings.filterwarnings("ignore")

# ============================================================
# User configuration
# ============================================================
USE_GOOGLE_DRIVE = True   # Set to False to run without mounting Google Drive.
PROJECT_NAME = 'Tourism_LLM_MultiCost_Planning'

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        BASE_DIR = Path('/content/drive/MyDrive/TravelerTripDataset/Outputs') / PROJECT_NAME
        print('Google Drive mode: ON')
    except Exception as e:
        print('Google Drive could not be mounted. Falling back to local outputs.')
        print(f'Reason: {e}')
        USE_GOOGLE_DRIVE = False

if not USE_GOOGLE_DRIVE:
    if Path('/content').exists():
        BASE_DIR = Path('/content/Outputs') / PROJECT_NAME
    else:
        BASE_DIR = Path.cwd() / 'Outputs' / PROJECT_NAME
    print('Google Drive mode: OFF')

FIG_DIR = BASE_DIR / 'figures'
TABLE_DIR = BASE_DIR / 'tables'
MODEL_DIR = BASE_DIR / 'models'
TEXT_DIR = BASE_DIR / 'text_outputs'
PROMPT_DIR = BASE_DIR / 'prompts_and_llm_responses'

for d in [BASE_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR, TEXT_DIR, PROMPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OUTPUT_SUMMARY = TEXT_DIR / 'outputs_summary.txt'
with open(OUTPUT_SUMMARY, 'w', encoding='utf-8') as f:
    f.write('Tourism LLM Multi-Cost Planning - Outputs Summary\n')
    f.write('=' * 70 + '\n\n')
    f.write(f'Google Drive mode: {USE_GOOGLE_DRIVE}\n')
    f.write(f'Base output directory: {BASE_DIR}\n\n')

print(f'Outputs will be saved in: {BASE_DIR}')


In [ ]:
dataset=pd.read_csv("/content/drive/MyDrive/TravelerTripDataset/dataset/TravelDetailsDataset.csv")

In [ ]:
dataset

In [ ]:
import pandas as pd

# Function to clean cost columns
def clean_cost_column(col):
    col = col.astype(str)
    col = col.str.replace(r'[^0-9.]', '', regex=True)  # remove $ USD etc.
    col = col.replace('', pd.NA)  # replace empty strings with NaN
    return pd.to_numeric(col, errors='coerce')  # convert safely

# Apply cleaning
dataset['Accommodation cost'] = clean_cost_column(dataset['Accommodation cost'])
dataset['Transportation cost'] = clean_cost_column(dataset['Transportation cost'])

# Check result
print(dataset[['Accommodation cost', 'Transportation cost']].head())
print(dataset.isna().sum())

In [ ]:
dataset

In [ ]:
# Convert Start date to datetime
dataset['Start date'] = pd.to_datetime(dataset['Start date'], errors='coerce')

# Extract month (you can use number or name)
dataset['Month'] = dataset['Start date'].dt.month   # numeric (1–12)

# OR if you prefer month name:
# dataset['Month'] = dataset['Start date'].dt.month_name()

# Drop unnecessary columns
dataset.drop(columns=['Start date', 'End date', 'Trip ID', 'Traveler name'], inplace=True)



In [ ]:
dataset

In [ ]:
## Here we will check the percentage of nan values present in each feature
## 1 -step make the list of features which has missing values
features_with_na=[features for features in dataset.columns if dataset[features].isnull().sum()>1]
## 2- step print the feature name and the percentage of missing values

for feature in features_with_na:
    print(feature, np.round(dataset[feature].isnull().mean(), 4),  '% missing values')

In [ ]:

# Separate features with missing values
features_with_na = [feature for feature in dataset.columns if dataset[feature].isnull().sum() > 0]

for feature in features_with_na:
    if dataset[feature].dtype == 'O':  # Object type = categorical
        most_frequent = dataset[feature].mode()[0]
        dataset[feature] = dataset[feature].fillna(most_frequent)
    else:  # Numerical feature
        median_value = dataset[feature].median()
        dataset[feature] = dataset[feature].fillna(median_value)

In [ ]:
## Here we will check the percentage of nan values present in each feature
## 1 -step make the list of features which has missing values
features_with_na=[features for features in dataset.columns if dataset[features].isnull().sum()>1]
## 2- step print the feature name and the percentage of missing values

for feature in features_with_na:
    print(feature, np.round(dataset[feature].isnull().mean(), 4),  '% missing values')

In [ ]:
#Categorical Variables
categorical_features=[feature for feature in dataset.columns if dataset[feature].dtypes=='O']
categorical_features

In [ ]:
for feature in categorical_features:
    unique_sorted = sorted(dataset[feature].dropna().unique())
    print(f"\nFeature: '{feature}' ({len(unique_sorted)} unique categories)")
    for idx, val in enumerate(unique_sorted):
        print(f"{idx}: {val}")

In [ ]:
dataset

In [ ]:
#import pandas as pd

# current size
#print(len(dataset))

# target size
#target_size = 1000

# bootstrap sampling (with replacement)
#dataset = dataset.sample(n=target_size, replace=True, random_state=42)

#print(len(dataset))

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------
# Step 1: Load dataset (assume already loaded)
# ---------------------------------------------------
df = dataset.copy()

# ---------------------------------------------------
# Step 2: Define augmentation settings
# ---------------------------------------------------
n_years_expand = 2        # how many synthetic cycles
noise_level = 0.03        # 3% controlled noise (safe range)
inflation_rate = 0.05     # 5% yearly increase (realistic assumption)

augmented_rows = []

# ---------------------------------------------------
# Step 3: Generate synthetic rows
# ---------------------------------------------------
for year in range(n_years_expand):

    for _, row in df.iterrows():
        new_row = row.copy()

        # simulate time effect (yearly inflation trend)
        growth_factor = (1 + inflation_rate) ** (year + 1)

        # apply controlled noise
        noise = np.random.normal(1, noise_level)

        # update cost columns safely
        new_row["Accommodation cost"] *= growth_factor * noise
        new_row["Transportation cost"] *= growth_factor * noise

        # recompute total cost
        new_row["Total Cost"] = (
            new_row["Accommodation cost"] + new_row["Transportation cost"]
        )

        augmented_rows.append(new_row)

# ---------------------------------------------------
# Step 4: Create expanded dataset
# ---------------------------------------------------
aug_df = pd.DataFrame(augmented_rows)

# Combine original + synthetic
dataset_expanded = pd.concat([df, aug_df], ignore_index=True)

# ---------------------------------------------------
# Step 5: Final check
# ---------------------------------------------------
print("Original size:", len(df))
print("Expanded size:", len(dataset_expanded))

dataset = dataset_expanded

In [ ]:
dataset

In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_features = [
    feature for feature in dataset.columns
    if dataset[feature].dtype == 'O'
]

encoders = {}  # store encoders for reproducibility

for feature in categorical_features:
    encoder = LabelEncoder()
    dataset[feature] = encoder.fit_transform(dataset[feature].astype(str))
    encoders[feature] = encoder

In [ ]:
dataset

**6. Leakage-safe Seasonal Intensity Score**
The Seasonal Intensity Score is computed as a historical destination–month cost intensity using only the training split:
SIS(d, m) = C_acc_train(d, m) / C_acc_train(d)
Where:


SIS(d, m) = Seasonal Intensity Score for destination d and month m


C_acc_train(d, m) = average accommodation cost for destination d in month m (computed using training data)


C_acc_train(d) = average accommodation cost for destination d across all months (computed using training data only)


For unseen destination–month combinations, a hierarchical fallback strategy is applied. First, destination-level averages are used when available; otherwise, global monthly averages are considered. If neither is available, a neutral value of 1.0 is assigned. This design ensures robustness while strictly preventing target leakage from the test set.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

# ---------------------------------------------------
# OUTPUT PATH
# ---------------------------------------------------
TABLE_DIR = Path("/content/drive/MyDrive/TravelerTripDataset/Outputs")
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------
# Step 1: Feature Engineering
# ---------------------------------------------------
dataset["Total Cost"] = dataset["Accommodation cost"] + dataset["Transportation cost"]

X = dataset.drop(columns=["Accommodation cost", "Transportation cost", "Total Cost"])
y_acc = dataset["Accommodation cost"]
y_tr = dataset["Transportation cost"]
y_total = dataset["Total Cost"]

# ---------------------------------------------------
# Step 2: Train-Test Split (ALIGNED)
# ---------------------------------------------------
X_train, X_test, y_acc_train, y_acc_test, y_tr_train, y_tr_test, y_total_train, y_total_test = train_test_split(
    X, y_acc, y_tr, y_total,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# ---------------------------------------------------
# Step 3: Build TRAIN DF for SIS computation ONLY
# ---------------------------------------------------
train_df = X_train.copy()
train_df["Accommodation cost"] = y_acc_train.values

# ---------------------------------------------------
# Step 4: FIT SIS (TRAIN ONLY → NO LEAKAGE)
# ---------------------------------------------------
global_mean = train_df["Accommodation cost"].mean()

dest_mean = train_df.groupby("Destination")["Accommodation cost"].mean()
month_mean = train_df.groupby("Month")["Accommodation cost"].mean()
dest_month_mean = train_df.groupby(["Destination", "Month"])["Accommodation cost"].mean()

sis_map = {}
for (d, m), val in dest_month_mean.items():
    denom = dest_mean.get(d, global_mean)
    sis_map[(d, m)] = float(val / denom) if denom > 0 else 1.0

dest_fallback = {
    d: float(dest_mean[d] / global_mean) if global_mean > 0 else 1.0
    for d in dest_mean.index
}

month_fallback = {
    m: float(val / global_mean) if global_mean > 0 else 1.0
    for m, val in month_mean.items()
}

# ---------------------------------------------------
# Step 5: APPLY SIS to TRAIN + TEST
# ---------------------------------------------------
def apply_sis(df):
    scores = []

    for _, r in df.iterrows():
        key = (r["Destination"], r["Month"])

        if key in sis_map:
            scores.append(sis_map[key])
        elif r["Destination"] in dest_fallback:
            scores.append(dest_fallback[r["Destination"]])
        elif r["Month"] in month_fallback:
            scores.append(month_fallback[r["Month"]])
        else:
            scores.append(1.0)

    df = df.copy()
    df["Seasonal_Intensity_Score"] = scores
    return df

X_train = apply_sis(X_train.copy())
X_test = apply_sis(X_test.copy())

# ---------------------------------------------------
# Step 6: SAVE CSV FILES
# ---------------------------------------------------
train_df_out = X_train.copy()
train_df_out["Accommodation cost"] = y_acc_train.values
train_df_out["Transportation cost"] = y_tr_train.values
train_df_out["Total cost"] = y_total_train.values

test_df_out = X_test.copy()
test_df_out["Accommodation cost"] = y_acc_test.values
test_df_out["Transportation cost"] = y_tr_test.values
test_df_out["Total cost"] = y_total_test.values

train_df_out.to_csv(TABLE_DIR / "train_with_sis.csv", index=False)
test_df_out.to_csv(TABLE_DIR / "test_with_sis.csv", index=False)

# ---------------------------------------------------
# Step 7: SIS SUMMARY CSV
# ---------------------------------------------------
summary_df = pd.DataFrame({
    "SIS_train_min": [train_df_out["Seasonal_Intensity_Score"].min()],
    "SIS_train_mean": [train_df_out["Seasonal_Intensity_Score"].mean()],
    "SIS_train_max": [train_df_out["Seasonal_Intensity_Score"].max()],
    "SIS_test_min": [test_df_out["Seasonal_Intensity_Score"].min()],
    "SIS_test_mean": [test_df_out["Seasonal_Intensity_Score"].mean()],
    "SIS_test_max": [test_df_out["Seasonal_Intensity_Score"].max()],
})

summary_df.to_csv(TABLE_DIR / "seasonal_intensity_summary.csv", index=False)

print("\nSIS computation complete!")
print("Saved to:", TABLE_DIR)

In [ ]:
X_train

In [ ]:
X_test

**8. Train one model per cost target with cross-validation**
Separate models are trained for accommodation, transportation, and total cost. Cross-validation is reported on the training set, and the final evaluation is conducted on the untouched test set.

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate

# ---------------------------------------------------
# TARGETS (MAKE SURE THESE EXIST)
# ---------------------------------------------------
targets = {
    "Accommodation cost": y_acc_train,
    "Transportation cost": y_tr_train,
    "Total Cost": y_total_train
}

# ---------------------------------------------------
# MODEL DIRECTORIES (define if not already)
# ---------------------------------------------------
MODEL_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# ---------------------------------------------------
# MODEL FACTORY (Random Forest instead of XGB)
# ---------------------------------------------------
def make_model():
    rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    return Pipeline(steps=[('model', rf)])

# ---------------------------------------------------
# CROSS VALIDATION
# ---------------------------------------------------
models = {}
cv_rows = []
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for target_name, y_train_target in targets.items():

    print(f"\nTraining target: {target_name}")

    pipe = make_model()

    scores = cross_validate(
        pipe,
        X_train,
        y_train_target,
        cv=cv,
        scoring=('r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error'),
        n_jobs=-1
    )

    cv_rows.append({
        'Target': target_name,
        'CV_R2_mean': scores['test_r2'].mean(),
        'CV_R2_std': scores['test_r2'].std(),
        'CV_MAE_mean': -scores['test_neg_mean_absolute_error'].mean(),
        'CV_RMSE_mean': -scores['test_neg_root_mean_squared_error'].mean(),
    })

    # ---------------------------------------------------
    # TRAIN FINAL MODEL
    # ---------------------------------------------------
    pipe.fit(X_train, y_train_target)
    models[target_name] = pipe

    # SAVE MODEL
    joblib.dump(pipe, MODEL_DIR / f"rf_pipeline_{target_name.replace(' ', '_')}.joblib")

# ---------------------------------------------------
# SAVE CV RESULTS
# ---------------------------------------------------
cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(TABLE_DIR / 'cross_validation_results.csv', index=False)

print("\nCross-validation results saved:")
display(cv_df)

9. Test-set evaluation and automatic

This evaluation uses only the untouched test set. The code computes RMSE manually as sqrt(MSE) for compatibility across scikit-learn versions. After reporting the metrics, the notebook prints a direct scientific interpretation so the author can immediately see whether the corrected performance is strong, moderate, or weak.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    max_error,
    explained_variance_score
)

import numpy as np
import pandas as pd

# ---------------------------------------------------
# MAPE FUNCTION (UNCHANGED)
# ---------------------------------------------------
def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


# ---------------------------------------------------
# PERFORMANCE LABEL (UNCHANGED)
# ---------------------------------------------------
def performance_label(r2_value):
    if pd.isna(r2_value):
        return 'undefined'
    if r2_value >= 0.80:
        return 'strong'
    if r2_value >= 0.60:
        return 'acceptable/moderate'
    if r2_value >= 0.40:
        return 'limited/moderate-low'
    return 'weak'


# ---------------------------------------------------
# TARGET MAPPING (IMPORTANT FIX)
# ---------------------------------------------------
y_test = {
    "Accommodation cost": y_acc_test,
    "Transportation cost": y_tr_test,
    "Total Cost": y_total_test
}

y_train = {
    "Accommodation cost": y_acc_train,
    "Transportation cost": y_tr_train,
    "Total Cost": y_total_train
}

targets = list(models.keys())

# ---------------------------------------------------
# EVALUATION
# ---------------------------------------------------
preds = {}
metric_rows = []

for target in targets:

    y_pred = models[target].predict(X_test)
    preds[target] = y_pred

    y_true = y_test[target].values.astype(float)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mean_y = np.mean(y_true)

    metric_rows.append({
        'Target': target,
        'R2': r2,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE_percent': mape(y_true, y_pred),
        'Median_AE': median_absolute_error(y_true, y_pred),
        'Max_Error': max_error(y_true, y_pred),
        'Explained_Variance': explained_variance_score(y_true, y_pred),
        'NRMSE': rmse / mean_y if mean_y != 0 else np.nan,
        'NMAE': mae / mean_y if mean_y != 0 else np.nan,
    })

metrics_df = pd.DataFrame(metric_rows)

metrics_path = TABLE_DIR / 'test_set_performance_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)

# ---------------------------------------------------
# INTERPRETATION (UNCHANGED LOGIC)
# ---------------------------------------------------
interpretation_lines = []
interpretation_lines.append('Interpretation of corrected test-set performance:')

for _, row in metrics_df.iterrows():
    label = performance_label(row['R2'])
    interpretation_lines.append(
        f"- {row['Target']}: R2={row['R2']:.3f}, RMSE={row['RMSE']:.2f}, "
        f"MAE={row['MAE']:.2f}, MAPE={row['MAPE_percent']:.2f}%. "
        f"This should be considered {label} predictive performance."
    )

avg_r2 = metrics_df['R2'].mean()

if avg_r2 < 0.60:
    interpretation_lines.append(
        "Overall conclusion: the leakage-safe and duplication-free evaluation does not support a claim of high predictive accuracy. "
        "The manuscript should not compare these results as superior to high-performing forecasting studies in the literature. "
        "Instead, the paper should be positioned as a decision-support and explainability framework operating under realistic uncertainty."
    )
else:
    interpretation_lines.append(
        "Overall conclusion: the predictive layer is at least moderate, but the paper should still avoid overclaiming and should emphasize decision support, interpretability, and reproducibility."
    )

interpretation_text = "\n".join(interpretation_lines)

print(interpretation_text)

with open(OUTPUT_SUMMARY, 'a', encoding='utf-8') as f:
    f.write('Test-set performance metrics\n')
    f.write(metrics_df.to_string(index=False))
    f.write('\n\n')
    f.write(interpretation_text)
    f.write('\n\n')

print(f"Saved metrics to: {metrics_path}")

display(metrics_df)

**Interpretation note for the manuscript**
If the corrected R² values are low or moderate, the manuscript should be revised accordingly. The safest and most scientifically defensible claim is not high-accuracy tourism cost forecasting, but rather transparent and actionable decision support under uncertainty. This distinction is essential for avoiding reviewer criticism.



In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
import xgboost as xgb

In [ ]:
import numpy as np
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    median_absolute_error, max_error, explained_variance_score
)

In [ ]:
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)

    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100

    med_ae = median_absolute_error(y_true, y_pred)
    max_e = max_error(y_true, y_pred)
    evs = explained_variance_score(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    # Normalized metrics
    nrmse = rmse / np.mean(y_true)
    nmae = mae / np.mean(y_true)

    return mse, rmse, mae, mape, med_ae, max_e, evs, r2, nrmse, nmae

In [ ]:
targets

In [ ]:
models

In [ ]:
model_acc=models[targets[0]]
model_tr=models[targets[1]]
model_total=models[targets[2]]

In [ ]:
print("\n===== ACCOMMODATION COST (Random Forest) =====")


y_pred_acc = model_acc.predict(X_test)

mse, rmse, mae, mape, med_ae, max_e, evs, r2, nrmse, nmae = regression_metrics(y_acc_test, y_pred_acc)

print("\n--- Random Forest Evaluation Metrics ---")
print("R² Score:", r2)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)
print("Mean Absolute Percentage Error (MAPE):", mape)
print("Median Absolute Error:", med_ae)
print("Max Error:", max_e)
print("Explained Variance Score:", evs)
print("Normalized RMSE (NRMSE):", nrmse)
print("Normalized MAE (NMAE):", nmae)

In [ ]:

print("\n===== TRANSPORTATION COST (Random Forest) =====")


y_pred_tr = model_tr.predict(X_test)

mse, rmse, mae, mape, med_ae, max_e, evs, r2, nrmse, nmae = regression_metrics(y_tr_test, y_pred_tr)

print("\n--- Random Forest Evaluation Metrics ---")
print("R² Score:", r2)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)
print("Mean Absolute Percentage Error (MAPE):", mape)
print("Median Absolute Error:", med_ae)
print("Max Error:", max_e)
print("Explained Variance Score:", evs)
print("Normalized RMSE (NRMSE):", nrmse)
print("Normalized MAE (NMAE):", nmae)

In [ ]:

print("\n===== TOTAL COST (Random Forest) =====")

y_pred_total = model_total.predict(X_test)

mse, rmse, mae, mape, med_ae, max_e, evs, r2, nrmse, nmae = regression_metrics(y_total_test, y_pred_total)

print("\n--- Random Forest Evaluation Metrics ---")
print("R² Score:", r2)
print("Mean Squared Error (MSE):", mse)
print("Root Mean Squared Error (RMSE):", rmse)
print("Mean Absolute Error (MAE):", mae)
print("Mean Absolute Percentage Error (MAPE):", mape)
print("Median Absolute Error:", med_ae)
print("Max Error:", max_e)
print("Explained Variance Score:", evs)
print("Normalized RMSE (NRMSE):", nrmse)
print("Normalized MAE (NMAE):", nmae)

In [ ]:
import joblib

save_path = "/content/drive/MyDrive/TravelerTripDataset/"

# Save Accommodation Cost model
joblib.dump(model_acc, save_path + "model_accommodation_cost.pkl")
print("✅ Accommodation cost model saved successfully!")

# Save Transportation Cost model
joblib.dump(model_tr, save_path + "model_transportation_cost.pkl")
print("✅ Transportation cost model saved successfully!")

# Save Total Cost model
joblib.dump(model_total, save_path + "model_total_cost.pkl")
print("✅ Total cost model saved successfully!")

In [ ]:
#when already saved, then next time do not click on train and save just load from here
import joblib

save_path = "/content/drive/MyDrive/TravelerTripDataset/"
model_acc = joblib.load(save_path + "model_accommodation_cost.pkl")
model_tr = joblib.load(save_path + "model_transportation_cost.pkl")
model_total = joblib.load(save_path + "model_total_cost.pkl")

In [ ]:
y_pred_acc = model_acc.predict(X_test)
y_pred_tr = model_tr.predict(X_test)
y_pred_total = model_total.predict(X_test)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ================= STYLE =================
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10
})

dpii = 300
n_line = 100  # number of samples to show

# ================= DATA =================
targets = [
    ("Accommodation Cost", y_acc_test, y_pred_acc),
    ("Transportation Cost", y_tr_test, y_pred_tr),
    ("Total Cost", y_total_test, y_pred_total)
]

# ================= MULTI-PLOT FIGURE =================
fig, axes = plt.subplots(len(targets), 1, figsize=(11, 3*len(targets)), dpi=dpii)

if len(targets) == 1:
    axes = [axes]

for ax, (name, y_true, y_pred) in zip(axes, targets):

    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)

    n = min(n_line, len(y_true))

    ax.plot(y_true[:n],
            label='Actual',
            linewidth=1.5,
            color='steelblue')

    ax.plot(y_pred[:n],
            linestyle='--',
            linewidth=1.3,
            label='Predicted',
            color='tomato')

    ax.set_title(f"{name} — test samples")
    ax.set_xlabel("Sample index")
    ax.set_ylabel("Cost")

    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)

# ================= GLOBAL TITLE =================


plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats

# ================= STYLE (from template) =================
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10
})

dpii = 300


def plot_model_diagnostics(y_true, y_pred, model_name):

    residuals = y_true - y_pred
    abs_error = np.abs(residuals)

    mn = min(y_true.min(), y_pred.min())
    mx = max(y_true.max(), y_pred.max())
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))

    # ================= FIGURE =================
    fig, axes = plt.subplots(2, 3, figsize=(12, 8), dpi=dpii)

    # -------- 1. Actual vs Predicted --------
    ax = axes[0, 0]
    ax.scatter(y_true, y_pred, alpha=0.6, s=20, color='steelblue')
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.2, label='Perfect prediction')

    # RMSE band (template style)
    ax.fill_between([mn, mx],
                    [mn-2*rmse, mx-2*rmse],
                    [mn+2*rmse, mx+2*rmse],
                    alpha=0.08, color='tomato', label='±2 RMSE band')

    ax.set_xlabel("Actual cost")
    ax.set_ylabel("Predicted cost")
    ax.set_title(f"{model_name}: Actual vs Predicted")
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)

    # -------- 2. Residuals vs Predicted --------
    ax = axes[0, 1]
    ax.scatter(y_pred, residuals, alpha=0.6, s=20, color='teal')
    ax.axhline(0, linestyle='--', color='red')
    ax.set_xlabel("Predicted cost")
    ax.set_ylabel("Residuals")
    ax.set_title("Residuals vs Predicted")
    ax.grid(True, linestyle='--', alpha=0.4)

    # -------- 3. Residual Distribution --------
    ax = axes[0, 2]
    sns.histplot(residuals, bins=30, kde=True, color='slateblue', ax=ax)
    ax.set_title("Residual Distribution")

    # -------- 4. Absolute Error Distribution --------
    ax = axes[1, 0]
    sns.histplot(abs_error, bins=30, kde=True, color='purple', ax=ax)
    ax.set_title("Absolute Error Distribution")

    # -------- 5. Boxplot --------
    ax = axes[1, 1]
    sns.boxplot(x=abs_error, color='skyblue', ax=ax)
    ax.set_title("Error Boxplot")

    # -------- 6. Q-Q Plot --------
    ax = axes[1, 2]
    stats.probplot(residuals, dist="norm", plot=ax)
    ax.set_title("Q-Q Plot")

    # ================= TITLE =================
    fig.suptitle(
        f"Figure: Diagnostic Analysis — {model_name}",
        fontweight='bold',
        fontsize=12
    )

    plt.tight_layout()
    plt.show()


# ================= APPLY TO YOUR MODELS =================

plot_model_diagnostics(y_acc_test.values, model_acc.predict(X_test), "Accommodation Cost")

plot_model_diagnostics(y_tr_test.values, model_tr.predict(X_test), "Transportation Cost")

plot_model_diagnostics(y_total_test.values, model_total.predict(X_test), "Total Cost")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ================= STYLE (unchanged from your template) =================
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10
})

dpii = 300


def plot_actual_vs_pred(ax, y_true, y_pred, model_name):

    residuals = y_true - y_pred
    rmse = np.sqrt(np.mean(residuals**2))

    mn = min(y_true.min(), y_pred.min())
    mx = max(y_true.max(), y_pred.max())

    # Scatter
    ax.scatter(y_true, y_pred, alpha=0.6, s=20, color='steelblue')

    # Perfect prediction line
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.2)

    # RMSE band
    ax.fill_between([mn, mx],
                    [mn - 2*rmse, mx - 2*rmse],
                    [mn + 2*rmse, mx + 2*rmse],
                    alpha=0.08, color='tomato')

    ax.set_title(model_name)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.grid(True, linestyle='--', alpha=0.4)


# ================= SINGLE ROW FIGURE =================
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=dpii)

plot_actual_vs_pred(
    axes[0],
    y_acc_test.values,
    model_acc.predict(X_test),
    "Accommodation Cost"
)

plot_actual_vs_pred(
    axes[1],
    y_tr_test.values,
    model_tr.predict(X_test),
    "Transportation Cost"
)

plot_actual_vs_pred(
    axes[2],
    y_total_test.values,
    model_total.predict(X_test),
    "Total Cost"
)

fig.suptitle(
    "Figure: Actual vs Predicted — Model Performance (With SIS)",
    fontweight='bold',
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

def error_category_analysis(X_test, y_test, y_pred, target_name):

    # ---------------- Reset index ----------------
    X_test_df = X_test.reset_index(drop=True)
    y_test = pd.Series(y_test).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)

    # ---------------- Create dataframe ----------------
    test_df = X_test_df.copy()
    test_df['Actual'] = y_test
    test_df['Predicted'] = y_pred

    # ---------------- Error calculations ----------------
    test_df['Error'] = test_df['Actual'] - test_df['Predicted']
    test_df['Absolute_Error'] = test_df['Error'].abs()
    test_df['Percentage_Error'] = (test_df['Absolute_Error'] / (test_df['Actual'] + 1e-8)) * 100

    # ---------------- Clean invalid values ----------------
    test_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    test_df.dropna(subset=['Actual', 'Predicted', 'Percentage_Error'], inplace=True)

    # ---------------- Error classification ----------------
    def classify_error(pct):
        if pct < 70:
            return 'Low Error (<70%)'
        elif pct <= 80:
            return 'Moderate Error (70-80%)'
        else:
            return 'High Error (>80%)'

    test_df['Error_Category'] = test_df['Percentage_Error'].apply(classify_error)

    # ---------------- Distribution (RAW) ----------------
    print(f"\n--- {target_name} Error Category Distribution ---")

    error_dist = test_df['Error_Category'].value_counts(normalize=True) * 100

    # ---------------- FORCE normalization to 100 ----------------
    error_dist = (error_dist / error_dist.sum()) * 100

    for cat, pct in error_dist.items():
        print(f"{cat}: {pct:.2f}%")

    print("Total:", error_dist.sum())

    return test_df

In [ ]:
import matplotlib.pyplot as plt

def plot_error_categories(test_df, title):

    # Get distribution
    error_dist = test_df['Error_Category'].value_counts(normalize=True) * 100
    error_dist = (error_dist / error_dist.sum()) * 100  # normalize to 100%

    # ✅ FIX: enforce ALL 3 categories always exist
    all_categories = [
        'Low Error (<70%)',
        'Moderate Error (70-80%)',
        'High Error (>80%)'
    ]

    error_dist = error_dist.reindex(all_categories, fill_value=0)

    categories = error_dist.index.tolist()
    percentages = error_dist.values.tolist()

    colors = ['#4C72B0', '#55A868', '#C44E52']

    plt.figure(figsize=(6, 4), dpi=200)
    bars = plt.bar(categories, percentages, color=colors)

    # Value labels
    for bar in bars:
        yval = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            yval + 1,
            f'{yval:.2f}%',
            ha='center',
            fontsize=9
        )

    plt.title(title, fontsize=11)
    plt.ylabel("Percentage (%)")
    plt.xlabel("Error Category")
    plt.ylim(0, 100)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
df_acc = error_category_analysis(
    X_test, y_acc_test, y_pred_acc,
    "Accommodation Cost"
)

In [ ]:
plot_error_categories(
    df_acc,
    "Accommodation Cost: Error Category Distribution"
)

In [ ]:
df_tr = error_category_analysis(
    X_test, y_tr_test, y_pred_tr,
    "Transportation Cost"
)

In [ ]:
plot_error_categories(
    df_tr,
    "Transportation Cost: Error Category Distribution"
)

In [ ]:
df_total = error_category_analysis(
    X_test, y_total_test, y_pred_total,
    "Total Cost"
)

In [ ]:
plot_error_categories(
    df_total,
    "Total Cost: Error Category Distribution"
)

In [ ]:
import warnings
import lime
import lime.lime_tabular
import numpy as np

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

# Convert training data
X_train_array = np.array(X_train)
feature_names = X_train.columns.tolist()

In [ ]:

explainer_acc = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_array,
    feature_names=feature_names,
    mode='regression'
)

def explain_acc(instance_index):
    instance = X_test.iloc[instance_index].values

    explanation = explainer_acc.explain_instance(
        data_row=instance,
        predict_fn=model_acc.predict
    )

    explanation.show_in_notebook(show_table=True)

In [ ]:

explainer_tr = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_array,
    feature_names=feature_names,
    mode='regression'
)

def explain_tr(instance_index):
    instance = X_test.iloc[instance_index].values

    explanation = explainer_tr.explain_instance(
        data_row=instance,
        predict_fn=model_tr.predict
    )

    explanation.show_in_notebook(show_table=True)

In [ ]:

explainer_total = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_array,
    feature_names=feature_names,
    mode='regression'
)

def explain_total(instance_index):
    instance = X_test.iloc[instance_index].values

    explanation = explainer_total.explain_instance(
        data_row=instance,
        predict_fn=model_total.predict
    )

    explanation.show_in_notebook(show_table=True)

In [ ]:
record_index=0
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
(y_pred_acc)

In [ ]:
record_index=1
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
record_index=2
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
record_index=3
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
record_index=10
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
record_index=20
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
record_index=40
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
record_index=55
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
record_index=65
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
record_index=70
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
record_index=83
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
record_index=30
print("Actual Accomodation Cost ($): " + str(y_acc_test.iloc[record_index]))
explain_acc(record_index)

print("Actual Transportation Cost ($): " + str(y_tr_test.iloc[record_index]))
explain_tr(record_index)

print("Actual Total Cost ($): " + str(y_total_test.iloc[record_index]))
explain_total(record_index)

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# CREATE FOLDERS
# -----------------------------
base_path = "/content/drive/MyDrive/TravelerTripDataset/xai/"

acc_path = os.path.join(base_path, "accommodation")
tr_path = os.path.join(base_path, "transportation")
total_path = os.path.join(base_path, "total")

os.makedirs(acc_path, exist_ok=True)
os.makedirs(tr_path, exist_ok=True)
os.makedirs(total_path, exist_ok=True)

print("✅ Folders created successfully!")

# -----------------------------
# GENERIC FUNCTION (REUSABLE)
# -----------------------------
def save_lime_explanations(explainer, model, y_test, X_test, save_dir, model_name, sample_indices):

    for i, idx in enumerate(sample_indices):

        actual = y_test.iloc[idx]
        predicted = model.predict(X_test.iloc[[idx]])[0]

        print(f"{model_name} | Index {idx} | Actual: {actual:.2f} | Pred: {predicted:.2f}")

        try:
            exp = explainer.explain_instance(
                X_test.iloc[idx].values,
                model.predict,
                num_features=len(X_test.columns)
            )

            # Convert to figure
            fig = exp.as_pyplot_figure()

            # Title
            fig.suptitle(
                f"{model_name} with SIS\nActual: {actual:.2f} | Predicted: {predicted:.2f}",
                fontsize=12,
                fontweight='bold'
            )

            # Color bars
            weights = [w for f, w in exp.as_list()]
            for j, bar in enumerate(fig.axes[0].patches):
                if j < len(weights):
                    bar.set_facecolor("darkblue" if weights[j] >= 0 else "crimson")

        except Exception as e:
            print(f"⚠️ LIME failed at index {idx}, using fallback: {e}")

            fig, ax = plt.subplots(figsize=(6,4), dpi=300)
            ax.bar(['Actual', 'Predicted'], [actual, predicted],
                   color=['darkblue', 'crimson'])
            ax.set_title(f"{model_name} (Fallback)\nIndex {idx}")

        # -----------------------------
        # SAVE FIGURE
        # -----------------------------
        filename = f"{model_name}_Index_{idx}_Actual_{actual:.1f}_Pred_{predicted:.1f}.png"
        fig.savefig(os.path.join(save_dir, filename), dpi=400, bbox_inches='tight')
        plt.close(fig)

In [ ]:
# Select sample indices (you can change this)
sample_indices = [1, 2, 3, 4, 5, 6, 7,8, 9,10,20,25,30,35,40,45,50,55,60,65,70,75,80,83]

# -----------------------------
# ACCOMMODATION
# -----------------------------
save_lime_explanations(
    explainer_acc,
    model_acc,
    y_acc_test,
    X_test,
    acc_path,
    "Accommodation",
    sample_indices
)

# -----------------------------
# TRANSPORTATION
# -----------------------------
save_lime_explanations(
    explainer_tr,
    model_tr,
    y_tr_test,
    X_test,
    tr_path,
    "Transportation",
    sample_indices
)

# -----------------------------
# TOTAL COST
# -----------------------------
save_lime_explanations(
    explainer_total,
    model_total,
    y_total_test,
    X_test,
    total_path,
    "Total",
    sample_indices
)

In [ ]:
import shap
import matplotlib.pyplot as plt

# (optional but recommended for clean plots)
shap.initjs()

In [ ]:
import shap
import matplotlib.pyplot as plt

# extract model from pipeline
rf_model = model_acc.named_steps['model']

# create explainer on actual model
explainer_acc = shap.TreeExplainer(rf_model)

# compute shap values
shap_values_acc = explainer_acc.shap_values(X_train)

# plot
plt.figure(figsize=(8, 6), dpi=300)
shap.summary_plot(shap_values_acc, X_train, show=False)
plt.title("SHAP Summary - Accommodation Cost with SIS")
plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt

# extract model from pipeline
rf_model = model_tr.named_steps['model']

# create explainer on actual model
explainer_acc = shap.TreeExplainer(rf_model)

# compute shap values
shap_values_acc = explainer_acc.shap_values(X_train)

# plot
plt.figure(figsize=(8, 6), dpi=300)
shap.summary_plot(shap_values_acc, X_train, show=False)
plt.title("SHAP Summary - Transportation Cost with SIS")
plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt

# extract model from pipeline
rf_model = model_total.named_steps['model']

# create explainer on actual model
explainer_acc = shap.TreeExplainer(rf_model)

# compute shap values
shap_values_acc = explainer_acc.shap_values(X_train)

# plot
plt.figure(figsize=(8, 6), dpi=300)
shap.summary_plot(shap_values_acc, X_train, show=False)
plt.title("SHAP Summary - Total Cost with SIS")
plt.show()

In [ ]:
pip install dice_ml

In [ ]:
dataset.columns

In [ ]:
import dice_ml
from dice_ml import Dice

In [ ]:
# Features user can change (agentic AI controls)
mutable_features = [
   'Destination', 'Duration (days)', 'Accommodation type', 'Transportation type',
       'Month', 'Seasonal_Intensity_Score'
]

# Features that MUST NOT change (real-world constraints)
immutable_features = [
    "Traveler age",
    "Traveler gender",
    "Traveler nationality"
]

In [ ]:
# Data object
d_acc = dice_ml.Data(
    dataframe=X_train.assign(Accommodation_Cost=y_acc_train),
    continuous_features=mutable_features,
    outcome_name='Accommodation_Cost'
)

# Model object
m_acc = dice_ml.Model(
    model=model_acc,
    backend="sklearn",
    model_type='regressor'
)

# Explainer
exp_acc = Dice(d_acc, m_acc, method="random")

In [ ]:

d_tr = dice_ml.Data(
    dataframe=X_train.assign(Transportation_Cost=y_tr_train),
    continuous_features=mutable_features,
    outcome_name='Transportation_Cost'
)

m_tr = dice_ml.Model(
    model=model_tr,
    backend="sklearn",
    model_type='regressor'
)

exp_tr = Dice(d_tr, m_tr, method="random")

In [ ]:

d_total = dice_ml.Data(
    dataframe=X_train.assign(Total_Cost=y_total_train),
    continuous_features=mutable_features,
    outcome_name='Total_Cost'
)

m_total = dice_ml.Model(
    model=model_total,
    backend="sklearn",
    model_type='regressor'
)

exp_total = Dice(d_total, m_total, method="random")

In [ ]:
def generate_cf(exp, X_test, index, desired_range, model_name):

    query_instance = X_test.iloc[index: index+1]

    cf = exp.generate_counterfactuals(
        query_instance,
        total_CFs=3,
        desired_range=desired_range
    )

    print(f"\n================ {model_name} COUNTERFACTUALS ================")
    cf.visualize_as_dataframe()

    return cf

In [ ]:
# Example: user wants cheaper transport
desired_range_tr = [200, 600]

cf_tr = generate_cf(
    exp_acc,
    X_test,
    index=1,
    desired_range=desired_range_tr,
    model_name="Accumodation Cost"
)

In [ ]:
# Example: user wants cheaper transport
desired_range_tr = [200, 600]

cf_tr = generate_cf(
    exp_tr,
    X_test,
    index=1,
    desired_range=desired_range_tr,
    model_name="Transportation Cost"
)

In [ ]:
# Example: user wants cheaper transport
desired_range_tr = [200, 600]

cf_tr = generate_cf(
    exp_total,
    X_test,
    index=1,
    desired_range=desired_range_tr,
    model_name="Transportation Cost"
)

In [ ]:
cf_tr = generate_cf(
    exp_acc,
    X_test,
    index=50,
    desired_range=desired_range_tr,
    model_name="Accumodation Cost"
)
cf_tr = generate_cf(
    exp_tr,
    X_test,
    index=50,
    desired_range=desired_range_tr,
    model_name="Transportation Cost"
)
cf_tr = generate_cf(
    exp_total,
    X_test,
    index=50,
    desired_range=desired_range_tr,
    model_name="Total Cost"
)

In [ ]:
# Example: user wants overall cheaper trip
desired_range_total = [200, 600]

cf_total = generate_cf(
    exp_total,
    X_test,
    index=50,
    desired_range=desired_range_total,
    model_name="Total Cost"
)

In [ ]:
# Example: user wants overall cheaper trip
desired_range_total = [1200, 2000]

cf_total = generate_cf(
    exp_total,
    X_test,
    index=50,
    desired_range=desired_range_total,
    model_name="Total Cost"
)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# ---------------- SLIDERS ----------------
acc_slider = widgets.IntSlider(
    value=800, min=100, max=3000, step=50,
    description='Accommodation Cost:', continuous_update=True
)

tr_slider = widgets.IntSlider(
    value=400, min=50, max=2000, step=50,
    description='Transportation Cost:', continuous_update=True
)

# ---------------- TEXT BOXES ----------------
total_text = widgets.IntText(
    value=1200,
    description='Total:'
)

index_text = widgets.IntText(
    value=0,
    description='Index:'
)

run_button = widgets.Button(
    description="Generate Counterfactuals",
    button_style='success'
)
run1_button = widgets.Button(
    description="LLM Responses",
    button_style='success'
)
output = widgets.Output()

In [ ]:
def update_total(change=None):
    total_text.value = acc_slider.value + tr_slider.value

# Link sliders to function
acc_slider.observe(update_total, names='value')
tr_slider.observe(update_total, names='value')

In [ ]:
def run_all(_):

    with output:
        output.clear_output()

        acc_base = acc_slider.value
        tr_base = tr_slider.value
        total_base = total_text.value
        idx = index_text.value

        desired_range_acc = [acc_base, acc_base + 100]
        desired_range_tr = [tr_base, tr_base + 50]
        desired_range_total = [total_base, total_base + 100]

        print("\n===== ACCOMMODATION =====")
        generate_cf(exp_acc, X_test, idx, desired_range_acc, "Accommodation Cost")

        print("\n===== TRANSPORTATION =====")
        generate_cf(exp_tr, X_test, idx, desired_range_tr, "Transportation Cost")

        print("\n===== TOTAL COST =====")
        generate_cf(exp_total, X_test, idx, desired_range_total, "Total Cost")

In [ ]:
run_button.on_click(run_all)
display(
    acc_slider,
    tr_slider,
    total_text,
    index_text,
    run_button,
    output
)

# initialize total value correctly
update_total()



In [ ]:
run_button.on_click(run_all)
display(
    acc_slider,
    tr_slider,
    total_text,
    index_text,
    run_button,
    output
)

# initialize total value correctly
update_total()

# initialize total value correctly
update_total()

In [ ]:
run_button.on_click(run_all)
display(
    acc_slider,
    tr_slider,
    total_text,
    index_text,
    run_button,
    output
)

# initialize total value correctly
update_total()

# initialize total value correctly
update_total()

In [ ]:
run_button.on_click(run_all)
display(
    acc_slider,
    tr_slider,
    total_text,
    index_text,
    run_button,
    output
)

# initialize total value correctly
update_total()

# initialize total value correctly
update_total()

In [ ]:
# ---------------- DESTINATION ----------------
destination_map = {
0:"Amsterdam",1:"Amsterdam, Netherlands",2:"Athens, Greece",3:"Auckland, New Zealand",
4:"Australia",5:"Bali",6:"Bali, Indonesia",7:"Bangkok",8:"Bangkok, Thai",9:"Bangkok, Thailand",
10:"Barcelona",11:"Barcelona, Spain",12:"Berlin, Germany",13:"Brazil",14:"Canada",
15:"Cancun, Mexico",16:"Cape Town",17:"Cape Town, SA",18:"Cape Town, South Africa",
19:"Dubai",20:"Dubai, United Arab Emirates",21:"Edinburgh, Scotland",22:"Egypt",
23:"France",24:"Greece",25:"Hawaii",26:"Honolulu, Hawaii",27:"Italy",28:"Japan",
29:"London",30:"London, UK",31:"Los Angeles, USA",32:"Marrakech, Morocco",
33:"Mexico",34:"New York",35:"New York City, USA",36:"New York, USA",
37:"Paris",38:"Paris, France",39:"Phnom Penh",40:"Phuket",41:"Phuket, Thai",
42:"Phuket, Thailand",43:"Rio de Janeiro",44:"Rio de Janeiro, Brazil",
45:"Rome",46:"Rome, Italy",47:"Santorini",48:"Seoul",49:"Seoul, South Korea",
50:"Spain",51:"Sydney",52:"Sydney, AUS",53:"Sydney, Aus",54:"Sydney, Australia",
55:"Thailand",56:"Tokyo",57:"Tokyo, Japan",58:"Vancouver, Canada"
}

# ---------------- GENDER ----------------
gender_map = {
0: "Female",
1: "Male"
}

# ---------------- NATIONALITY ----------------
nationality_map = {
0:"American",1:"Australian",2:"Brazil",3:"Brazilian",4:"British",5:"Cambodia",
6:"Canada",7:"Canadian",8:"China",9:"Chinese",10:"Dutch",11:"Emirati",
12:"French",13:"German",14:"Germany",15:"Greece",16:"Hong Kong",
17:"Indian",18:"Indonesian",19:"Italian",20:"Italy",21:"Japan",22:"Japanese",
23:"Korean",24:"Mexican",25:"Moroccan",26:"New Zealander",27:"Scottish",
28:"Singapore",29:"South African",30:"South Korea",31:"South Korean",
32:"Spain",33:"Spanish",34:"Taiwan",35:"Taiwanese",36:"UK",37:"USA",
38:"United Arab Emirates",39:"United Kingdom",40:"Vietnamese"
}

# ---------------- ACCOMMODATION ----------------
accommodation_map = {
0:"Airbnb",1:"Guesthouse",2:"Hostel",3:"Hotel",
4:"Resort",5:"Riad",6:"Vacation rental",7:"Villa"
}

# ---------------- TRANSPORT ----------------
transport_map = {
0:"Airplane",1:"Bus",2:"Car",3:"Car rental",
4:"Ferry",5:"Flight",6:"Plane",7:"Subway",8:"Train"
}

In [ ]:
feature_value_map = {}

def build_feature_map(dataset, categorical_features):
    global feature_value_map

    for feature in categorical_features:
        unique_sorted = sorted(dataset[feature].dropna().unique())

        feature_value_map[feature] = {
            idx: val for idx, val in enumerate(unique_sorted)
        }

    print("✅ Feature mapping created successfully!")

In [ ]:
def decode_feature(feature_name, value):

    maps = {
        "Destination": destination_map,
        "Traveler gender": gender_map,
        "Traveler nationality": nationality_map,
        "Accommodation type": accommodation_map,
        "Transportation type": transport_map
    }

    try:
        value = int(value)
        return maps[feature_name].get(value, f"Unknown ({value})")

    except:
        return f"Invalid ({value})"

In [ ]:
import google.generativeai as genai

MY_KEY = ""
genai.configure(api_key=MY_KEY)


def ask_gemini_tourism_agent(
    record_index,
    X_test,
    y_pred_acc,
    y_pred_tr,
    y_pred_total,
    model_name="gemini-2.5-flash"
):

    model = genai.GenerativeModel(model_name)

    row = X_test.iloc[record_index]

    # -----------------------------
    # 🔥 PROPER DECODING (IMPORTANT FIX)
    # -----------------------------
    destination = decode_feature("Destination", row["Destination"])
    nationality = decode_feature("Traveler nationality", row["Traveler nationality"])
    gender = decode_feature("Traveler gender", row["Traveler gender"])
    acc_type = decode_feature("Accommodation type", row["Accommodation type"])
    transport_type = decode_feature("Transportation type", row["Transportation type"])

    month = row.get("month", "Unknown")
    duration = row.get("Duration (days)", "Unknown")
    age = row.get("Traveler age", "Unknown")

    seasonal_intensity = row.get("Seasonal_Intensity_Score", "Unknown")

    # -----------------------------
    # PREDICTIONS
    # -----------------------------
    acc_cost = float(y_pred_acc[record_index])
    tr_cost = float(y_pred_tr[record_index])
    total_cost = float(y_pred_total[record_index])

    cost_level = "Low" if total_cost < 1000 else "Medium" if total_cost < 2000 else "High"

    # -----------------------------
    # LLM PROMPT (CLEAN + CORRECT)
    # -----------------------------
    prompt = f"""
You are a Tourism Cost Advisor AI.

Keep all answers:
- VERY SHORT
- BULLET POINTS ONLY
- MAX 1 LINE PER POINT
- NO EXPLANATIONS OR STORIES

-----------------------------------
TRAVEL INFO
-----------------------------------
Destination: {destination}
Nationality: {nationality}
Gender: {gender}
Accommodation: {acc_type}
Transport: {transport_type}
Month: {month}
Season: {seasonal_intensity}
Duration: {duration}
Age: {age}

-----------------------------------
COSTS
-----------------------------------
Accommodation: {acc_cost:.2f}
Transport: {tr_cost:.2f}
Total: {total_cost:.2f}
Level: {cost_level}

-----------------------------------
TASK
-----------------------------------
- 3 cost drivers only
- 3 saving suggestions only
- 1 optimized plan
- 1 final decision (Proceed/Modify/Delay)
- 1 confidence level (High/Medium/Low)

-----------------------------------
OUTPUT FORMAT (STRICT)
-----------------------------------
Drivers:
- ...
- ...
- ...

Suggestions:
- ...
- ...
- ...

Plan:
- ...

Decision:
- ...

Confidence:
- ...
"""
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Error: {e}"

In [ ]:
build_feature_map(dataset, categorical_features)
# already trained models
y_pred_acc = model_acc.predict(X_test)
y_pred_tr = model_tr.predict(X_test)
y_pred_total = model_total.predict(X_test)

In [ ]:
result = ask_gemini_tourism_agent(
    record_index=0,
    X_test=X_test,
    y_pred_acc=y_pred_acc,
    y_pred_tr=y_pred_tr,
    y_pred_total=y_pred_total
)

print(result)

In [ ]:
result = ask_gemini_tourism_agent(
    record_index=1,
    X_test=X_test,
    y_pred_acc=y_pred_acc,
    y_pred_tr=y_pred_tr,
    y_pred_total=y_pred_total
)

print(result)

In [ ]:
result = ask_gemini_tourism_agent(
    record_index=83,
    X_test=X_test,
    y_pred_acc=y_pred_acc,
    y_pred_tr=y_pred_tr,
    y_pred_total=y_pred_total
)

print(result)

In [ ]:
result = ask_gemini_tourism_agent(
    record_index=70,
    X_test=X_test,
    y_pred_acc=y_pred_acc,
    y_pred_tr=y_pred_tr,
    y_pred_total=y_pred_total
)

print(result)

In [ ]:
result = ask_gemini_tourism_agent(
    record_index=50,
    X_test=X_test,
    y_pred_acc=y_pred_acc,
    y_pred_tr=y_pred_tr,
    y_pred_total=y_pred_total
)

print(result)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_rel, wilcoxon

# -----------------------------
# STEP 1: SELECT FIRST 15 RECORDS
# -----------------------------
N = 10
X_eval = X_test.iloc[:N].reset_index(drop=True)

y_acc_eval = y_acc_test.iloc[:N].reset_index(drop=True)
y_tr_eval = y_tr_test.iloc[:N].reset_index(drop=True)
y_total_eval = y_total_test.iloc[:N].reset_index(drop=True)

pred_acc_eval = y_pred_acc[:N]
pred_tr_eval = y_pred_tr[:N]
pred_total_eval = y_pred_total[:N]

# -----------------------------
# STEP 2: LLM CALL FUNCTION WRAPPER
# -----------------------------
def run_llm(index):
    response = ask_gemini_tourism_agent(
        record_index=index,
        X_test=X_test,
        y_pred_acc=y_pred_acc,
        y_pred_tr=y_pred_tr,
        y_pred_total=y_pred_total
    )
    return response

# -----------------------------
# STEP 3: PARSE LLM OUTPUT
# -----------------------------
def parse_llm_output(text):
    decision = None
    confidence = None

    lines = text.split("\n")

    for i, line in enumerate(lines):

        # detect Decision
        if "Decision:" in line:
            # next line contains value
            if i + 1 < len(lines):
                decision = lines[i+1].replace("-", "").strip()

        # detect Confidence
        if "Confidence:" in line:
            if i + 1 < len(lines):
                confidence = lines[i+1].replace("-", "").strip()

    return decision, confidence

# -----------------------------
# STEP 4: BUILD EVALUATION TABLE
# -----------------------------
records = []

for i in range(N):
    llm_output = run_llm(i)
    print("--------------- " + str(i) + " --------------------")
    print(llm_output)
    print("-----------------------------------")
    decision, confidence = parse_llm_output(llm_output)

    pred_total = pred_total_eval[i]

    # Cost level
    if pred_total < 1000:
        cost_level = "Low"
    elif pred_total < 2000:
        cost_level = "Medium"
    else:
        cost_level = "High"

    # alignment rule (simple research proxy)
    if cost_level == "Low" and decision == "Proceed":
        alignment = 1
    elif cost_level == "High" and decision == "Delay":
        alignment = 1
    else:
        alignment = 0

    # confidence score mapping
    conf_score = 1 if confidence == "High" else 0.5 if confidence == "Medium" else 0

    records.append({
        "Index": i,
        "Pred_Total": pred_total,
        "Cost_Level": cost_level,
        "Decision": decision,
        "Confidence": confidence,
        "Confidence_Score": conf_score,
        "Decision_Score": alignment
    })

eval_df = pd.DataFrame(records)

print(eval_df)

In [ ]:
eval_df

In [ ]:
eval_df

In [ ]:
t_stat, p_value = ttest_rel(eval_df["Pred_Total"], eval_df["Confidence_Score"])

print("\n===== PAIRED T-TEST =====")
print("T-statistic:", t_stat)
print("P-value:", p_value)

In [ ]:
w_stat, p_value_w = wilcoxon(eval_df["Pred_Total"], eval_df["Confidence_Score"])

print("\n===== WILCOXON TEST =====")
print("Statistic:", w_stat)
print("P-value:", p_value_w)

In [ ]:
plt.figure(figsize=(5,4), dpi=200)
eval_df['Decision_Score'].value_counts().plot(kind='bar', color='#4C72B0')
plt.title("LLM Decision Alignment Score")
plt.xlabel("Score (1=Correct, 0=Mismatch)")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--')
plt.show()

In [ ]:
plt.figure(figsize=(5,4), dpi=200)
eval_df['Confidence'].value_counts().plot(kind='bar', color='#55A868')
plt.title("LLM Confidence Distribution")
plt.grid(axis='y', linestyle='--')
plt.show()

In [ ]:
plt.figure(figsize=(6,4), dpi=200)
eval_df.boxplot(column='Pred_Total', by='Decision')
plt.title("Cost Distribution by LLM Decision")
plt.suptitle("")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(6,4), dpi=200)
sns.countplot(data=eval_df, x='Cost_Level', hue='Decision')
plt.title("Decision vs Cost Level")
plt.grid(axis='y', linestyle='--')
plt.show()

In [ ]:
plt.figure(figsize=(6,4), dpi=200)
plt.scatter(eval_df['Pred_Total'], eval_df['Confidence_Score'], color='#C44E52')
plt.title("Confidence vs Total Cost")
plt.xlabel("Predicted Cost")
plt.ylabel("Confidence Score")
plt.grid(True)
plt.show()

In [ ]:
display(
    acc_slider,
    tr_slider,
    total_text,
    index_text,
    run_button,
    run1_button,
    output
)